# Lesson Iceberg (extended)

Урок проходится сверху вниз. Ноутбук расширен разделами про branches, tags, WAP и CDC.


## 1. Импорты и SparkSession

Spark + Iceberg runtime + REST catalog + S3FileIO под MinIO.


In [1]:
import os
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql import functions as F


spark = (
    SparkSession.builder
    .appName("LessonIceberg")
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.3,"
        "org.apache.spark:spark-avro_2.12:3.5.3,"
        "com.databricks:spark-xml_2.12:0.18.0,"
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1,"
        "org.apache.iceberg:iceberg-aws-bundle:1.6.1",
    )
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.iceberg.type", "rest")
    .config("spark.sql.catalog.iceberg.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.iceberg.warehouse", "s3://lakehouse")
    .config("spark.sql.catalog.iceberg.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.iceberg.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.iceberg.s3.path-style-access", "true")
    .config("spark.sql.catalog.iceberg.s3.region", "us-east-1")
    .config("spark.sql.catalog.iceberg.s3.access-key-id", os.getenv("MINIO_ROOT_USER"))
    .config("spark.sql.catalog.iceberg.s3.secret-access-key", os.getenv("MINIO_ROOT_PASSWORD"))
    .config("spark.driver.extraJavaOptions", "-Daws.region=us-east-1")
    .config("spark.executor.extraJavaOptions", "-Daws.region=us-east-1")
    .getOrCreate()
)

hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", os.getenv("MINIO_ROOT_USER"))
hadoop_conf.set("fs.s3a.secret.key", os.getenv("MINIO_ROOT_PASSWORD"))
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.path.style.access", "true")

spark.sparkContext.setLogLevel("WARN")


## 2. Рабочий namespace


In [2]:
CATALOG = "iceberg"
NAMESPACE = "halltape"

spark.sql(f"CREATE NAMESPACE IF NOT EXISTS {CATALOG}.{NAMESPACE}")
spark.sql(f"SHOW NAMESPACES IN {CATALOG}").show(truncate=False)


+---------+
|namespace|
+---------+
|halltape |
+---------+



## 3. Подготовка таблиц


In [3]:
lesson_tables = [
    "orders",
    "orders_rollback_demo",
    "orders_schema_evolution",
    "events_partition_demo",
    "cow_orders",
    "mor_orders",
    "sort_orders",
    "stats_demo",
    "orders_branch_demo",
    "wap_orders",
    "users_cdc_demo",
]

for table_name in lesson_tables:
    spark.sql(f"DROP TABLE IF EXISTS {CATALOG}.{NAMESPACE}.{table_name} PURGE")

spark.sql(f"SHOW TABLES IN {CATALOG}.{NAMESPACE}").show(truncate=False)

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
+---------+---------+-----------+



## 4. Создание Iceberg-таблицы `orders`


In [4]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{NAMESPACE}.orders (
    order_id INT,
    customer STRING,
    city STRING,
    category STRING,
    amount DOUBLE,
    event_ts TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(event_ts))
TBLPROPERTIES ('format-version' = '2')
""")

spark.sql(f"DESCRIBE TABLE EXTENDED {CATALOG}.{NAMESPACE}.orders").show(truncate=False)

+----------------------------+-------------------------+-------+
|col_name                    |data_type                |comment|
+----------------------------+-------------------------+-------+
|order_id                    |int                      |NULL   |
|customer                    |string                   |NULL   |
|city                        |string                   |NULL   |
|category                    |string                   |NULL   |
|amount                      |double                   |NULL   |
|event_ts                    |timestamp                |NULL   |
|                            |                         |       |
|# Partitioning              |                         |       |
|Part 0                      |days(event_ts)           |       |
|                            |                         |       |
|# Metadata Columns          |                         |       |
|_spec_id                    |int                      |       |
|_partition              

In [5]:
orders_data = [
    (1001, "Alice", "Moscow", "electronics", 1299.90, datetime(2026, 4, 10, 10, 15)),
    (1002, "Bob", "Kazan", "books", 42.50, datetime(2026, 4, 10, 12, 5)),
    (1003, "Carrol", "Saint Petersburg", "home", 310.00, datetime(2026, 4, 11, 9, 30)),
    (1004, "Diana", "Sochi", "sport", 560.40, datetime(2026, 4, 11, 19, 45)),
    (1005, "Egor", "Moscow", "games", 74.99, datetime(2026, 4, 12, 14, 20)),
    (1006, "Fatima", "Kazan", "electronics", 2200.00, datetime(2026, 4, 12, 16, 0)),
]

orders_df = spark.createDataFrame(
    orders_data,
    ["order_id", "customer", "city", "category", "amount", "event_ts"]
)

orders_df.show(truncate=False)


+--------+--------+----------------+-----------+------+-------------------+
|order_id|customer|city            |category   |amount|event_ts           |
+--------+--------+----------------+-----------+------+-------------------+
|1001    |Alice   |Moscow          |electronics|1299.9|2026-04-10 10:15:00|
|1002    |Bob     |Kazan           |books      |42.5  |2026-04-10 12:05:00|
|1003    |Carrol  |Saint Petersburg|home       |310.0 |2026-04-11 09:30:00|
|1004    |Diana   |Sochi           |sport      |560.4 |2026-04-11 19:45:00|
|1005    |Egor    |Moscow          |games      |74.99 |2026-04-12 14:20:00|
|1006    |Fatima  |Kazan           |electronics|2200.0|2026-04-12 16:00:00|
+--------+--------+----------------+-----------+------+-------------------+



In [6]:
orders_df.writeTo(f"{CATALOG}.{NAMESPACE}.orders").append()

spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders ORDER BY order_id").show(truncate=False)


+--------+--------+----------------+-----------+------+-------------------+
|order_id|customer|city            |category   |amount|event_ts           |
+--------+--------+----------------+-----------+------+-------------------+
|1001    |Alice   |Moscow          |electronics|1299.9|2026-04-10 10:15:00|
|1002    |Bob     |Kazan           |books      |42.5  |2026-04-10 12:05:00|
|1003    |Carrol  |Saint Petersburg|home       |310.0 |2026-04-11 09:30:00|
|1004    |Diana   |Sochi           |sport      |560.4 |2026-04-11 19:45:00|
|1005    |Egor    |Moscow          |games      |74.99 |2026-04-12 14:20:00|
|1006    |Fatima  |Kazan           |electronics|2200.0|2026-04-12 16:00:00|
+--------+--------+----------------+-----------+------+-------------------+



In [7]:
spark.sql(f"""
SELECT committed_at, snapshot_id, parent_id, operation
FROM {CATALOG}.{NAMESPACE}.orders.snapshots
ORDER BY committed_at
""").show(truncate=False)

+-----------------------+-------------------+---------+---------+
|committed_at           |snapshot_id        |parent_id|operation|
+-----------------------+-------------------+---------+---------+
|2026-10-04 12:37:25.405|1696978258226539989|NULL     |append   |
+-----------------------+-------------------+---------+---------+



## 5. Что физически появилось в S3

`lakehouse/<namespace>/orders/data/` — parquet, `lakehouse/<namespace>/orders/metadata/` — avro/json.


In [9]:
row = spark.sql(f"SELECT file_path FROM {CATALOG}.{NAMESPACE}.orders.files LIMIT 1").first()
parquet_file_path = row["file_path"].replace("s3://", "s3a://")
print("Первый parquet-файл:", parquet_file_path)
spark.read.parquet(parquet_file_path).show(truncate=False)

Первый parquet-файл: s3a://lakehouse/halltape/orders/data/event_ts_day=2026-04-10/00000-32-a92e8c5f-d76e-42b1-87a7-f885a8996f8a-0-00001.parquet
+--------+--------+------+-----------+------+-------------------+
|order_id|customer|city  |category   |amount|event_ts           |
+--------+--------+------+-----------+------+-------------------+
|1001    |Alice   |Moscow|electronics|1299.9|2026-04-10 10:15:00|
|1002    |Bob     |Kazan |books      |42.5  |2026-04-10 12:05:00|
+--------+--------+------+-----------+------+-------------------+



In [11]:
row = spark.sql(f"SELECT path FROM {CATALOG}.{NAMESPACE}.orders.manifests LIMIT 1").first()
manifest_file_path = row["path"].replace("s3://", "s3a://")
print("Manifest file:", manifest_file_path)
spark.read.format("avro").load(manifest_file_path).show(truncate=False)

Manifest file: s3a://lakehouse/halltape/orders/metadata/6e9f9f62-7119-4d9b-9a18-dce6700b9419-m0.avro
+------+-------------------+---------------+--------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|status|snapshot_id        |sequence_number|file_sequence_number|data_file                                                                       

In [13]:
row = spark.sql(f"SELECT manifest_list FROM {CATALOG}.{NAMESPACE}.orders.snapshots LIMIT 1").first()
manifest_list_path = row["manifest_list"].replace("s3://", "s3a://")
print("Manifest list:", manifest_list_path)
spark.read.format("avro").load(manifest_list_path).show(truncate=False)

Manifest list: s3a://lakehouse/halltape/orders/metadata/snap-1696978258226539989-1-6e9f9f62-7119-4d9b-9a18-dce6700b9419.avro
+------------------------------------------------------------------------------------+---------------+-----------------+-------+---------------+-------------------+-------------------+-----------------+--------------------+-------------------+----------------+-------------------+------------------+----------------------------------------------+------------+
|manifest_path                                                                       |manifest_length|partition_spec_id|content|sequence_number|min_sequence_number|added_snapshot_id  |added_files_count|existing_files_count|deleted_files_count|added_rows_count|existing_rows_count|deleted_rows_count|partitions                                    |key_metadata|
+------------------------------------------------------------------------------------+---------------+-----------------+-------+---------------+-----------

In [15]:
row = spark.sql(f"SELECT file FROM {CATALOG}.{NAMESPACE}.orders.metadata_log_entries ORDER BY latest_snapshot_id DESC LIMIT 1").first()
metadata_json_path = row["file"].replace("s3://", "s3a://")
print("Metadata json:", metadata_json_path)
(spark.read.json(metadata_json_path)
 .select("format-version", "location", "current-snapshot-id", "last-sequence-number")
 .show(truncate=False))

Metadata json: s3a://lakehouse/halltape/orders/metadata/00001-eb2f69a5-d60e-4136-8212-46d17e03824b.metadata.json
+--------------+------------------------------+-------------------+--------------------+
|format-version|location                      |current-snapshot-id|last-sequence-number|
+--------------+------------------------------+-------------------+--------------------+
|2             |s3://lakehouse/halltape/orders|1696978258226539989|1                   |
+--------------+------------------------------+-------------------+--------------------+



## 6. Append и новые snapshots


In [16]:
orders_append_df = spark.createDataFrame(
    [
        (1007, "Gleb", "Novosibirsk", "books", 23.90, datetime(2026, 4, 13, 11, 10)),
        (1008, "Hannah", "Moscow", "home", 415.30, datetime(2026, 4, 13, 18, 25)),
        (1009, "Ivan", "Vladivostok", "travel", 980.00, datetime(2026, 4, 14, 8, 40)),
    ],
    ["order_id", "customer", "city", "category", "amount", "event_ts"]
)

orders_append_df.writeTo(f"{CATALOG}.{NAMESPACE}.orders").append()

spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders ORDER BY order_id").show(truncate=False)
spark.sql(f"""
SELECT committed_at, snapshot_id, parent_id, operation
FROM {CATALOG}.{NAMESPACE}.orders.snapshots
ORDER BY committed_at
""").show(truncate=False)


+--------+--------+----------------+-----------+------+-------------------+
|order_id|customer|city            |category   |amount|event_ts           |
+--------+--------+----------------+-----------+------+-------------------+
|1001    |Alice   |Moscow          |electronics|1299.9|2026-04-10 10:15:00|
|1002    |Bob     |Kazan           |books      |42.5  |2026-04-10 12:05:00|
|1003    |Carrol  |Saint Petersburg|home       |310.0 |2026-04-11 09:30:00|
|1004    |Diana   |Sochi           |sport      |560.4 |2026-04-11 19:45:00|
|1005    |Egor    |Moscow          |games      |74.99 |2026-04-12 14:20:00|
|1006    |Fatima  |Kazan           |electronics|2200.0|2026-04-12 16:00:00|
|1007    |Gleb    |Novosibirsk     |books      |23.9  |2026-04-13 11:10:00|
|1008    |Hannah  |Moscow          |home       |415.3 |2026-04-13 18:25:00|
|1009    |Ivan    |Vladivostok     |travel     |980.0 |2026-04-14 08:40:00|
+--------+--------+----------------+-----------+------+-------------------+

+----------

## 7. Time travel


In [17]:
snapshot_rows = spark.sql(f"""
SELECT committed_at, snapshot_id
FROM {CATALOG}.{NAMESPACE}.orders.snapshots
ORDER BY committed_at
""").collect()

first_snapshot_id = snapshot_rows[0]["snapshot_id"]
latest_snapshot_id = snapshot_rows[-1]["snapshot_id"]
first_snapshot_time = snapshot_rows[0]["committed_at"].strftime("%Y-%m-%d %H:%M:%S.%f")
latest_snapshot_time = snapshot_rows[-1]["committed_at"].strftime("%Y-%m-%d %H:%M:%S.%f")

print("first_snapshot_id =", first_snapshot_id)
print("first_snapshot_time =", first_snapshot_time)
print("latest_snapshot_id =", latest_snapshot_id)
print("latest_snapshot_time =", latest_snapshot_time)

first_snapshot_id = 1696978258226539989
first_snapshot_time = 2026-10-04 12:37:25.405000
latest_snapshot_id = 4632536869969288809
latest_snapshot_time = 2026-10-04 13:30:19.611000


In [18]:
spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders VERSION AS OF {first_snapshot_id} ORDER BY order_id").show(truncate=False)
spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders TIMESTAMP AS OF '{first_snapshot_time}' ORDER BY order_id").show(truncate=False)

+--------+--------+----------------+-----------+------+-------------------+
|order_id|customer|city            |category   |amount|event_ts           |
+--------+--------+----------------+-----------+------+-------------------+
|1001    |Alice   |Moscow          |electronics|1299.9|2026-04-10 10:15:00|
|1002    |Bob     |Kazan           |books      |42.5  |2026-04-10 12:05:00|
|1003    |Carrol  |Saint Petersburg|home       |310.0 |2026-04-11 09:30:00|
|1004    |Diana   |Sochi           |sport      |560.4 |2026-04-11 19:45:00|
|1005    |Egor    |Moscow          |games      |74.99 |2026-04-12 14:20:00|
|1006    |Fatima  |Kazan           |electronics|2200.0|2026-04-12 16:00:00|
+--------+--------+----------------+-----------+------+-------------------+

+--------+--------+----------------+-----------+------+-------------------+
|order_id|customer|city            |category   |amount|event_ts           |
+--------+--------+----------------+-----------+------+-------------------+
|1001    |A

In [19]:
spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders VERSION AS OF {latest_snapshot_id} ORDER BY order_id").show(truncate=False)
spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders TIMESTAMP AS OF '{latest_snapshot_time}' ORDER BY order_id").show(truncate=False)


+--------+--------+----------------+-----------+------+-------------------+
|order_id|customer|city            |category   |amount|event_ts           |
+--------+--------+----------------+-----------+------+-------------------+
|1001    |Alice   |Moscow          |electronics|1299.9|2026-04-10 10:15:00|
|1002    |Bob     |Kazan           |books      |42.5  |2026-04-10 12:05:00|
|1003    |Carrol  |Saint Petersburg|home       |310.0 |2026-04-11 09:30:00|
|1004    |Diana   |Sochi           |sport      |560.4 |2026-04-11 19:45:00|
|1005    |Egor    |Moscow          |games      |74.99 |2026-04-12 14:20:00|
|1006    |Fatima  |Kazan           |electronics|2200.0|2026-04-12 16:00:00|
|1007    |Gleb    |Novosibirsk     |books      |23.9  |2026-04-13 11:10:00|
|1008    |Hannah  |Moscow          |home       |415.3 |2026-04-13 18:25:00|
|1009    |Ivan    |Vladivostok     |travel     |980.0 |2026-04-14 08:40:00|
+--------+--------+----------------+-----------+------+-------------------+

+--------+-

## 8. Rollback to snapshot


In [20]:
spark.sql(f"CREATE TABLE IF NOT EXISTS {CATALOG}.{NAMESPACE}.orders_rollback_demo USING iceberg AS SELECT * FROM {CATALOG}.{NAMESPACE}.orders")

spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders_rollback_demo ORDER BY order_id").show(truncate=False)


+--------+--------+----------------+-----------+------+-------------------+
|order_id|customer|city            |category   |amount|event_ts           |
+--------+--------+----------------+-----------+------+-------------------+
|1001    |Alice   |Moscow          |electronics|1299.9|2026-04-10 10:15:00|
|1002    |Bob     |Kazan           |books      |42.5  |2026-04-10 12:05:00|
|1003    |Carrol  |Saint Petersburg|home       |310.0 |2026-04-11 09:30:00|
|1004    |Diana   |Sochi           |sport      |560.4 |2026-04-11 19:45:00|
|1005    |Egor    |Moscow          |games      |74.99 |2026-04-12 14:20:00|
|1006    |Fatima  |Kazan           |electronics|2200.0|2026-04-12 16:00:00|
|1007    |Gleb    |Novosibirsk     |books      |23.9  |2026-04-13 11:10:00|
|1008    |Hannah  |Moscow          |home       |415.3 |2026-04-13 18:25:00|
|1009    |Ivan    |Vladivostok     |travel     |980.0 |2026-04-14 08:40:00|
+--------+--------+----------------+-----------+------+-------------------+



In [21]:
spark.sql(f"DELETE FROM {CATALOG}.{NAMESPACE}.orders_rollback_demo WHERE amount < 100")
spark.sql(f"UPDATE {CATALOG}.{NAMESPACE}.orders_rollback_demo SET city = 'Kaliningrad' WHERE order_id = 1004")

spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders_rollback_demo ORDER BY order_id").show(truncate=False)
spark.sql(f"""
SELECT committed_at, snapshot_id, parent_id, operation
FROM {CATALOG}.{NAMESPACE}.orders_rollback_demo.snapshots
ORDER BY committed_at
""").show(truncate=False)


+--------+--------+----------------+-----------+------+-------------------+
|order_id|customer|city            |category   |amount|event_ts           |
+--------+--------+----------------+-----------+------+-------------------+
|1001    |Alice   |Moscow          |electronics|1299.9|2026-04-10 10:15:00|
|1003    |Carrol  |Saint Petersburg|home       |310.0 |2026-04-11 09:30:00|
|1004    |Diana   |Kaliningrad     |sport      |560.4 |2026-04-11 19:45:00|
|1006    |Fatima  |Kazan           |electronics|2200.0|2026-04-12 16:00:00|
|1008    |Hannah  |Moscow          |home       |415.3 |2026-04-13 18:25:00|
|1009    |Ivan    |Vladivostok     |travel     |980.0 |2026-04-14 08:40:00|
+--------+--------+----------------+-----------+------+-------------------+

+-----------------------+-------------------+-------------------+---------+
|committed_at           |snapshot_id        |parent_id          |operation|
+-----------------------+-------------------+-------------------+---------+
|2026-10-04

In [23]:
rollback_before_id = spark.sql(f"""
SELECT snapshot_id
FROM {CATALOG}.{NAMESPACE}.orders_rollback_demo.snapshots
ORDER BY committed_at
LIMIT 1
""").first()["snapshot_id"]

print("snapshot для rollback =", rollback_before_id)

spark.sql(f"CALL {CATALOG}.system.rollback_to_snapshot('{NAMESPACE}.orders_rollback_demo', {rollback_before_id})").show(truncate=False)

spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders_rollback_demo ORDER BY order_id").show(truncate=False)
spark.sql(f"SELECT name, type, snapshot_id FROM {CATALOG}.{NAMESPACE}.orders_rollback_demo.refs").show(truncate=False)

spark.sql(f"""
SELECT committed_at, snapshot_id, parent_id, operation
FROM {CATALOG}.{NAMESPACE}.orders_rollback_demo.snapshots
ORDER BY committed_at
""").show(truncate=False)


snapshot для rollback = 6015295809421664604
+--------------------+-------------------+
|previous_snapshot_id|current_snapshot_id|
+--------------------+-------------------+
|6015295809421664604 |6015295809421664604|
+--------------------+-------------------+

+--------+--------+----------------+-----------+------+-------------------+
|order_id|customer|city            |category   |amount|event_ts           |
+--------+--------+----------------+-----------+------+-------------------+
|1001    |Alice   |Moscow          |electronics|1299.9|2026-04-10 10:15:00|
|1002    |Bob     |Kazan           |books      |42.5  |2026-04-10 12:05:00|
|1003    |Carrol  |Saint Petersburg|home       |310.0 |2026-04-11 09:30:00|
|1004    |Diana   |Sochi           |sport      |560.4 |2026-04-11 19:45:00|
|1005    |Egor    |Moscow          |games      |74.99 |2026-04-12 14:20:00|
|1006    |Fatima  |Kazan           |electronics|2200.0|2026-04-12 16:00:00|
|1007    |Gleb    |Novosibirsk     |books      |23.9  |2

## 9. Служебные таблицы Iceberg


In [24]:
spark.sql(f"DESCRIBE TABLE EXTENDED {CATALOG}.{NAMESPACE}.orders").show(truncate=False)
spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders.history").show(truncate=False)
spark.sql(f"""
SELECT committed_at, snapshot_id, parent_id, operation, manifest_list
FROM {CATALOG}.{NAMESPACE}.orders.snapshots
ORDER BY committed_at
""").show(truncate=False)
spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders.manifests").show(truncate=False)
spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders.all_manifests").show(truncate=False)
spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders.metadata_log_entries").show(truncate=False)
spark.sql(f"""
SELECT content, file_path, record_count, file_size_in_bytes
FROM {CATALOG}.{NAMESPACE}.orders.files
ORDER BY file_path
""").show(truncate=False)
spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders.partitions").show(truncate=False)


+----------------------------+-------------------------+-------+
|col_name                    |data_type                |comment|
+----------------------------+-------------------------+-------+
|order_id                    |int                      |NULL   |
|customer                    |string                   |NULL   |
|city                        |string                   |NULL   |
|category                    |string                   |NULL   |
|amount                      |double                   |NULL   |
|event_ts                    |timestamp                |NULL   |
|                            |                         |       |
|# Partitioning              |                         |       |
|Part 0                      |days(event_ts)           |       |
|                            |                         |       |
|# Metadata Columns          |                         |       |
|_spec_id                    |int                      |       |
|_partition              

## 10. Schema evolution


In [25]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{NAMESPACE}.orders_schema_evolution (
    order_id INT,
    customer STRING,
    event_ts TIMESTAMP
)
USING iceberg
TBLPROPERTIES ('format-version' = '2')
""")

spark.createDataFrame(
    [(2001, "Alice", datetime(2026, 4, 15, 10, 0)), (2002, "Bob", datetime(2026, 4, 15, 11, 0))],
    ["order_id", "customer", "event_ts"]
).writeTo(f"{CATALOG}.{NAMESPACE}.orders_schema_evolution").append()

spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders_schema_evolution ORDER BY order_id").show(truncate=False)


+--------+--------+-------------------+
|order_id|customer|event_ts           |
+--------+--------+-------------------+
|2001    |Alice   |2026-04-15 10:00:00|
|2002    |Bob     |2026-04-15 11:00:00|
+--------+--------+-------------------+



In [28]:
spark.sql(f"ALTER TABLE {CATALOG}.{NAMESPACE}.orders_schema_evolution ADD COLUMN priority STRING COMMENT 'Приоритет обработки заказа'")
spark.sql(f"ALTER TABLE {CATALOG}.{NAMESPACE}.orders_schema_evolution RENAME COLUMN customer TO customer_name")

spark.createDataFrame(
    [(2003, "high", "Carrol", datetime(2026, 4, 16, 9, 15)), (2004, "normal", "Diana", datetime(2026, 4, 16, 9, 45))],
    ["order_id", "priority", "customer_name", "event_ts"]
).writeTo(f"{CATALOG}.{NAMESPACE}.orders_schema_evolution").append()

spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders_schema_evolution ORDER BY order_id").show(truncate=False)

spark.sql(f"ALTER TABLE {CATALOG}.{NAMESPACE}.orders_schema_evolution DROP COLUMN priority")

spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders_schema_evolution ORDER BY order_id").show(truncate=False)
spark.sql(f"""
SELECT committed_at, snapshot_id, operation
FROM {CATALOG}.{NAMESPACE}.orders_schema_evolution.snapshots
ORDER BY committed_at
""").show(truncate=False)


AnalysisException: [FIELDS_ALREADY_EXISTS] Cannot add column, because `priority` already exists in "STRUCT<order_id: INT, customer_name: STRING, event_ts: TIMESTAMP, priority: STRING COMMENT 'Приоритет обработки заказа'>".; line 1 pos 0;
AddColumns [QualifiedColType(None,priority,StringType,true,Some(Приоритет обработки заказа),None,None)]
+- ResolvedTable org.apache.iceberg.spark.SparkCatalog@2f62d078, halltape.orders_schema_evolution, iceberg.halltape.orders_schema_evolution, [order_id#2051, customer_name#2052, event_ts#2053, priority#2054]


## 11. Hidden partitions и partition evolution


In [29]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{NAMESPACE}.events_partition_demo (
    event_id INT,
    user_id INT,
    category STRING,
    event_ts TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(event_ts))
TBLPROPERTIES ('format-version' = '2')
""")

spark.createDataFrame(
    [
        (1, 10, "search", datetime(2026, 4, 10, 10, 0)),
        (2, 10, "view", datetime(2026, 4, 10, 10, 1)),
        (3, 11, "cart", datetime(2026, 4, 11, 12, 30)),
        (4, 12, "purchase", datetime(2026, 4, 12, 18, 45)),
    ],
    ["event_id", "user_id", "category", "event_ts"]
).writeTo(f"{CATALOG}.{NAMESPACE}.events_partition_demo").append()

spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.events_partition_demo.partitions").show(truncate=False)


+------------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+----------------------+------------------------+
|partition   |spec_id|record_count|file_count|total_data_file_size_in_bytes|position_delete_record_count|position_delete_file_count|equality_delete_record_count|equality_delete_file_count|last_updated_at       |last_updated_snapshot_id|
+------------+-------+------------+----------+-----------------------------+----------------------------+--------------------------+----------------------------+--------------------------+----------------------+------------------------+
|{2026-04-10}|0      |2           |1         |1236                         |0                           |0                         |0                           |0                         |2026-10-04 16:18:07.67|8860601338751066607     |
|{2026-04-11}|0      |1           |1         |1178  

In [30]:
filtered_events = spark.sql(f"""
SELECT *
FROM {CATALOG}.{NAMESPACE}.events_partition_demo
WHERE event_ts >= TIMESTAMP '2026-04-10 00:00:00'
  AND event_ts < TIMESTAMP '2026-04-11 00:00:00'
""")

filtered_events.explain("formatted")
filtered_events.show(truncate=False)


== Physical Plan ==
* ColumnarToRow (2)
+- BatchScan iceberg.halltape.events_partition_demo (1)


(1) BatchScan iceberg.halltape.events_partition_demo
Output [4]: [event_id#2156, user_id#2157, category#2158, event_ts#2159]
iceberg.halltape.events_partition_demo (branch=null) [filters=event_ts IS NOT NULL, event_ts >= 1775779200000000, event_ts < 1775865600000000, groupedBy=]

(2) ColumnarToRow [codegen id : 1]
Input [4]: [event_id#2156, user_id#2157, category#2158, event_ts#2159]


+--------+-------+--------+-------------------+
|event_id|user_id|category|event_ts           |
+--------+-------+--------+-------------------+
|1       |10     |search  |2026-04-10 10:00:00|
|2       |10     |view    |2026-04-10 10:01:00|
+--------+-------+--------+-------------------+



In [32]:
#spark.sql(f"ALTER TABLE {CATALOG}.{NAMESPACE}.events_partition_demo ADD PARTITION FIELD category")

# spark.createDataFrame(
#     [
#         (5, 13, "search", datetime(2026, 4, 13, 9, 0)),
#         (6, 13, "purchase", datetime(2026, 4, 13, 9, 20)),
#     ],
#     ["event_id", "user_id", "category", "event_ts"]
# ).writeTo(f"{CATALOG}.{NAMESPACE}.events_partition_demo").append()

spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.events_partition_demo.partitions").show(truncate=False, vertical=True)
spark.sql(f"SELECT spec_id, content, file_path, record_count FROM {CATALOG}.{NAMESPACE}.events_partition_demo.files").show(truncate=False)


-RECORD 0------------------------------------------------
 partition                     | {2026-04-13, purchase}  
 spec_id                       | 1                       
 record_count                  | 1                       
 file_count                    | 1                       
 total_data_file_size_in_bytes | 1207                    
 position_delete_record_count  | 0                       
 position_delete_file_count    | 0                       
 equality_delete_record_count  | 0                       
 equality_delete_file_count    | 0                       
 last_updated_at               | 2026-10-04 16:53:30.544 
 last_updated_snapshot_id      | 4258551175191321576     
-RECORD 1------------------------------------------------
 partition                     | {2026-04-11, NULL}      
 spec_id                       | 0                       
 record_count                  | 1                       
 file_count                    | 1                       
 total_data_fi

## 12. Copy-on-Write и Merge-on-Read


In [33]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{NAMESPACE}.cow_orders (
    id INT,
    category STRING,
    amount DOUBLE
)
USING iceberg
TBLPROPERTIES ('format-version' = '2')
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{NAMESPACE}.mor_orders (
    id INT,
    category STRING,
    amount DOUBLE
)
USING iceberg
TBLPROPERTIES (
    'format-version' = '2',
    'write.delete.mode' = 'merge-on-read',
    'write.update.mode' = 'merge-on-read',
    'write.merge.mode' = 'merge-on-read'
)
""")


DataFrame[]

In [34]:
orders_for_update = spark.range(1, 2001).select(
    F.col("id").cast("int"),
    F.when((F.col("id") % 3) == 0, F.lit("electronics"))
     .when((F.col("id") % 3) == 1, F.lit("books"))
     .otherwise(F.lit("home")).alias("category"),
    (F.col("id") * F.lit(1.25)).cast("double").alias("amount")
).repartition(8)

orders_for_update.writeTo(f"{CATALOG}.{NAMESPACE}.cow_orders").append()
orders_for_update.writeTo(f"{CATALOG}.{NAMESPACE}.mor_orders").append()

spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.cow_orders").show(5)
spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.mor_orders").show(5)


+---+-----------+------+
| id|   category|amount|
+---+-----------+------+
| 89|       home|111.25|
| 24|electronics|  30.0|
| 15|electronics| 18.75|
|114|electronics| 142.5|
| 66|electronics|  82.5|
+---+-----------+------+
only showing top 5 rows

+---+-----------+------+
| id|   category|amount|
+---+-----------+------+
| 89|       home|111.25|
| 24|electronics|  30.0|
| 15|electronics| 18.75|
|114|electronics| 142.5|
| 66|electronics|  82.5|
+---+-----------+------+
only showing top 5 rows



In [35]:
spark.sql(f"DELETE FROM {CATALOG}.{NAMESPACE}.cow_orders WHERE id <= 5")
spark.sql(f"DELETE FROM {CATALOG}.{NAMESPACE}.mor_orders WHERE id <= 5")

print("CoW files")
spark.sql(f"SELECT content, count(*) AS files, sum(record_count) AS records FROM {CATALOG}.{NAMESPACE}.cow_orders.files GROUP BY content ORDER BY content").show(truncate=False)

print("MoR files")
spark.sql(f"SELECT content, count(*) AS files, sum(record_count) AS records FROM {CATALOG}.{NAMESPACE}.mor_orders.files GROUP BY content ORDER BY content").show(truncate=False)


CoW files
+-------+-----+-------+
|content|files|records|
+-------+-----+-------+
|0      |5    |1995   |
+-------+-----+-------+

MoR files
+-------+-----+-------+
|content|files|records|
+-------+-----+-------+
|0      |8    |2000   |
|1      |1    |5      |
+-------+-----+-------+



In [36]:
spark.sql(f"""
    SELECT file_path, record_count
    FROM {CATALOG}.{NAMESPACE}.cow_orders.files
""").show(truncate=False)

+------------------------------------------------------------------------------------------------------+------------+
|file_path                                                                                             |record_count|
+------------------------------------------------------------------------------------------------------+------------+
|s3://lakehouse/halltape/cow_orders/data/00000-233-85ad735f-ff14-4b6e-a48a-c3df24b52def-0-00001.parquet|996         |
|s3://lakehouse/halltape/cow_orders/data/00000-196-5ff0aa0e-80a0-4d0e-b19c-a9cfc2522ade-0-00001.parquet|253         |
|s3://lakehouse/halltape/cow_orders/data/00003-199-5ff0aa0e-80a0-4d0e-b19c-a9cfc2522ade-0-00001.parquet|246         |
|s3://lakehouse/halltape/cow_orders/data/00004-200-5ff0aa0e-80a0-4d0e-b19c-a9cfc2522ade-0-00001.parquet|247         |
|s3://lakehouse/halltape/cow_orders/data/00006-202-5ff0aa0e-80a0-4d0e-b19c-a9cfc2522ade-0-00001.parquet|253         |
+-------------------------------------------------------

## 13. rewrite_data_files и сортировка


In [37]:
spark.sql(f"""
CALL {CATALOG}.system.rewrite_data_files(
    table => '{NAMESPACE}.mor_orders',
    options => map(
        'target-file-size-bytes', '67108864',
        'max-concurrent-file-group-rewrites', '3'
    )
)
""").show(truncate=False)

spark.sql(f"""
SELECT content, file_path, record_count, file_format, equality_ids
FROM {CATALOG}.{NAMESPACE}.mor_orders.files
ORDER BY content, file_path
""").show(truncate=False)

+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|8                         |1                     |16945                |0                      |
+--------------------------+----------------------+---------------------+-----------------------+

+-------+------------------------------------------------------------------------------------------------------------+------------+-----------+------------+
|content|file_path                                                                                                   |record_count|file_format|equality_ids|
+-------+------------------------------------------------------------------------------------------------------------+------------+-----------+------------+
|0      |s3://lakehouse/halltape/mor_o

In [39]:
# spark.sql(f"""
# CREATE TABLE {CATALOG}.{NAMESPACE}.sort_orders (
#     id INT,
#     bucket STRING,
#     amount DOUBLE
# )
# USING iceberg
# TBLPROPERTIES ('format-version' = '2')
# """)

# spark.range(1, 5001).select(
#     F.col("id").cast("int"),
#     F.concat(F.lit("bucket_"), (F.col("id") % 7)).alias("bucket"),
#     (F.rand(seed=42) * 1000).cast("double").alias("amount")
# ).repartition(16).writeTo(f"{CATALOG}.{NAMESPACE}.sort_orders").append()

spark.sql(f"SELECT file_path, record_count, lower_bounds, upper_bounds FROM {CATALOG}.{NAMESPACE}.sort_orders.files").show(truncate=False, vertical=True)

spark.sql(f"""
CALL {CATALOG}.system.rewrite_data_files(
    table => '{NAMESPACE}.sort_orders',
    strategy => 'sort',
    sort_order => 'id ASC',
    options => map('target-file-size-bytes', '1048576')
)
""").show(truncate=False)

spark.sql(f"SELECT file_path, record_count, lower_bounds, upper_bounds FROM {CATALOG}.{NAMESPACE}.sort_orders.files").show(truncate=False, vertical=True)


-RECORD 0---------------------------------------------------------------------------------------------------------------
 file_path    | s3://lakehouse/halltape/sort_orders/data/00000-277-749a46c1-c638-4134-bc1d-af9bd5951a25-0-00001.parquet 
 record_count | 5000                                                                                                    
 lower_bounds | {1 -> [01 00 00 00], 2 -> [62 75 63 6B 65 74 5F 30], 3 -> [00 58 58 CE 5F 46 8C 3F]}                    
 upper_bounds | {1 -> [88 13 00 00], 2 -> [62 75 63 6B 65 74 5F 36], 3 -> [CB 91 10 60 DB 3F 8F 40]}                    

+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|0                         |0                     |0                    |0                      |
+-

In [40]:
spark.sql(f"SELECT file_path FROM {CATALOG}.{NAMESPACE}.sort_orders.files").show(truncate=False)

+-------------------------------------------------------------------------------------------------------+
|file_path                                                                                              |
+-------------------------------------------------------------------------------------------------------+
|s3://lakehouse/halltape/sort_orders/data/00000-277-749a46c1-c638-4134-bc1d-af9bd5951a25-0-00001.parquet|
+-------------------------------------------------------------------------------------------------------+



## 14. Статистики и expiration

In [41]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{NAMESPACE}.stats_demo (
    id INT,
    customer STRING,
    city STRING,
    amount DOUBLE,
    comment STRING
)
USING iceberg
TBLPROPERTIES ('format-version' = '2')
""")

spark.sql(f"""
ALTER TABLE {CATALOG}.{NAMESPACE}.stats_demo SET TBLPROPERTIES (
    'write.metadata.metrics.default' = 'counts',
    'write.metadata.metrics.column.id' = 'full',
    'write.metadata.metrics.column.amount' = 'full',
    'write.metadata.metrics.column.comment' = 'none',
    'write.metadata.delete-after-commit.enabled' = 'true',
    'write.metadata.previous-versions-max' = '3'
)
""")

for batch in range(3):
    spark.range(batch * 100, (batch + 1) * 100).select(
        F.col("id").cast("int"),
        F.concat(F.lit("customer_"), F.col("id")).alias("customer"),
        F.when((F.col("id") % 2) == 0, F.lit("Moscow")).otherwise(F.lit("Kazan")).alias("city"),
        (F.col("id") * F.lit(3.14)).cast("double").alias("amount"),
        F.concat(F.lit("debug payload "), F.col("id")).alias("comment"),
    ).repartition(4).writeTo(f"{CATALOG}.{NAMESPACE}.stats_demo").append()

spark.sql(f"SELECT record_count, null_value_counts, lower_bounds, upper_bounds FROM {CATALOG}.{NAMESPACE}.stats_demo.files").show(truncate=False)
spark.sql(f"SELECT committed_at, snapshot_id, operation FROM {CATALOG}.{NAMESPACE}.stats_demo.snapshots ORDER BY committed_at").show(truncate=False)

spark.sql(f"""
CALL {CATALOG}.system.expire_snapshots(
  table => '{NAMESPACE}.stats_demo',
  older_than => TIMESTAMP '2100-01-01 00:00:00',
  retain_last => 1
)
""")
spark.sql(f"SELECT committed_at, snapshot_id, operation FROM {CATALOG}.{NAMESPACE}.stats_demo.snapshots ORDER BY committed_at").show(truncate=False)

+------------+--------------------------------+----------------------------------------------------+----------------------------------------------------+
|record_count|null_value_counts               |lower_bounds                                        |upper_bounds                                        |
+------------+--------------------------------+----------------------------------------------------+----------------------------------------------------+
|28          |{1 -> 0, 2 -> 0, 3 -> 0, 4 -> 0}|{4 -> [0A D7 A3 70 3D D2 83 40], 1 -> [CA 00 00 00]}|{4 -> [E2 7A 14 AE 47 D9 8C 40], 1 -> [26 01 00 00]}|
|25          |{1 -> 0, 2 -> 0, 3 -> 0, 4 -> 0}|{4 -> [90 C2 F5 28 5C EB 83 40], 1 -> [CB 00 00 00]}|{4 -> [71 3D 0A D7 A3 24 8D 40], 1 -> [29 01 00 00]}|
|21          |{1 -> 0, 2 -> 0, 3 -> 0, 4 -> 0}|{4 -> [00 00 00 00 00 A0 83 40], 1 -> [C8 00 00 00]}|{4 -> [EC 51 B8 1E 85 0B 8D 40], 1 -> [28 01 00 00]}|
|26          |{1 -> 0, 2 -> 0, 3 -> 0, 4 -> 0}|{4 -> [85 EB 51 B8 1E B9 83 4

## 15. Branches (ветки)

Ветка — это отдельная именованная ссылка на snapshot. Можно писать в ветку, не трогая `main`, а потом переключить `main` (fast-forward) или просто читать ветку через `VERSION AS OF 'branch_name'`.

Ключевые команды:
- `ALTER TABLE t CREATE BRANCH name`
- `ALTER TABLE t DROP BRANCH name`
- `ALTER TABLE t REPLACE BRANCH name AS OF VERSION <snapshot_id>`
- запись в ветку: `df.writeTo(t).option("branch", "name").append()`
- чтение: `SELECT * FROM t VERSION AS OF 'name'`

In [42]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{NAMESPACE}.orders_branch_demo (
    order_id INT,
    customer STRING,
    city STRING,
    amount DOUBLE,
    event_ts TIMESTAMP
)
USING iceberg
TBLPROPERTIES ('format-version' = '2')
""")

base_data = [
    (1, "Alice", "Moscow", 100.0, datetime(2026, 5, 1, 10, 0)),
    (2, "Bob", "Kazan", 200.0, datetime(2026, 5, 1, 11, 0)),
    (3, "Carrol", "Sochi", 300.0, datetime(2026, 5, 2, 12, 0)),
]

spark.createDataFrame(base_data, ["order_id", "customer", "city", "amount", "event_ts"]) \
    .writeTo(f"{CATALOG}.{NAMESPACE}.orders_branch_demo").append()

spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders_branch_demo ORDER BY order_id").show(truncate=False)


+--------+--------+------+------+-------------------+
|order_id|customer|city  |amount|event_ts           |
+--------+--------+------+------+-------------------+
|1       |Alice   |Moscow|100.0 |2026-05-01 10:00:00|
|2       |Bob     |Kazan |200.0 |2026-05-01 11:00:00|
|3       |Carrol  |Sochi |300.0 |2026-05-02 12:00:00|
+--------+--------+------+------+-------------------+



In [43]:
spark.sql(f"ALTER TABLE {CATALOG}.{NAMESPACE}.orders_branch_demo CREATE BRANCH audit")

spark.sql(f"SELECT name, type, snapshot_id FROM {CATALOG}.{NAMESPACE}.orders_branch_demo.refs").show(truncate=False)


+-----+------+-------------------+
|name |type  |snapshot_id        |
+-----+------+-------------------+
|main |BRANCH|8430032323435406341|
|audit|BRANCH|8430032323435406341|
+-----+------+-------------------+



In [44]:
new_orders = [
    (4, "Diana", "Moscow", 400.0, datetime(2026, 5, 3, 9, 0)),
    (5, "Egor", "Kazan", 500.0, datetime(2026, 5, 3, 10, 0)),
]
audit_df = spark.createDataFrame(new_orders, ["order_id", "customer", "city", "amount", "event_ts"])

audit_df.writeTo(f"{CATALOG}.{NAMESPACE}.orders_branch_demo").option("branch", "audit").append()

print("main:")
spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders_branch_demo ORDER BY order_id").show(truncate=False)

print("audit:")
spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders_branch_demo VERSION AS OF 'audit' ORDER BY order_id").show(truncate=False)


main:
+--------+--------+------+------+-------------------+
|order_id|customer|city  |amount|event_ts           |
+--------+--------+------+------+-------------------+
|1       |Alice   |Moscow|100.0 |2026-05-01 10:00:00|
|2       |Bob     |Kazan |200.0 |2026-05-01 11:00:00|
|3       |Carrol  |Sochi |300.0 |2026-05-02 12:00:00|
|4       |Diana   |Moscow|400.0 |2026-05-03 09:00:00|
|5       |Egor    |Kazan |500.0 |2026-05-03 10:00:00|
+--------+--------+------+------+-------------------+

audit:
+--------+--------+------+------+-------------------+
|order_id|customer|city  |amount|event_ts           |
+--------+--------+------+------+-------------------+
|1       |Alice   |Moscow|100.0 |2026-05-01 10:00:00|
|2       |Bob     |Kazan |200.0 |2026-05-01 11:00:00|
|3       |Carrol  |Sochi |300.0 |2026-05-02 12:00:00|
+--------+--------+------+------+-------------------+



Быстрый forward `main` на `audit`. Так часто делают, когда ветка прошла проверку.


In [45]:
spark.sql(f"SELECT name, type, snapshot_id FROM {CATALOG}.{NAMESPACE}.orders_branch_demo.refs").show(truncate=False)

+-----+------+-------------------+
|name |type  |snapshot_id        |
+-----+------+-------------------+
|main |BRANCH|7322931409751006332|
|audit|BRANCH|8430032323435406341|
+-----+------+-------------------+



In [46]:
spark.sql(f"CALL {CATALOG}.system.fast_forward('{NAMESPACE}.orders_branch_demo', 'main', 'audit')").show(truncate=False)

spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders_branch_demo ORDER BY order_id").show(truncate=False)
spark.sql(f"SELECT name, type, snapshot_id FROM {CATALOG}.{NAMESPACE}.orders_branch_demo.refs").show(truncate=False)


IllegalArgumentException: Cannot fast-forward: main is not an ancestor of audit

## 16. Tags (теги)

Тег — это тоже ссылка на snapshot, но неизменяемая: он не двигается, пока его не заменят вручную. Удобно вешать версии «v1», «v2» и читать их потом.


In [47]:
spark.sql(f"ALTER TABLE {CATALOG}.{NAMESPACE}.orders_branch_demo CREATE TAG v1")

spark.sql(f"SELECT name, type, snapshot_id FROM {CATALOG}.{NAMESPACE}.orders_branch_demo.refs").show(truncate=False)


+-----+------+-------------------+
|name |type  |snapshot_id        |
+-----+------+-------------------+
|main |BRANCH|7322931409751006332|
|v1   |TAG   |7322931409751006332|
|audit|BRANCH|8430032323435406341|
+-----+------+-------------------+



In [48]:
spark.sql(f"DELETE FROM {CATALOG}.{NAMESPACE}.orders_branch_demo WHERE order_id = 1")

print("main после DELETE:")
spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders_branch_demo ORDER BY order_id").show(truncate=False)

print("чтение из tag v1:")
spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.orders_branch_demo VERSION AS OF 'v1' ORDER BY order_id").show(truncate=False)


main после DELETE:
+--------+--------+------+------+-------------------+
|order_id|customer|city  |amount|event_ts           |
+--------+--------+------+------+-------------------+
|2       |Bob     |Kazan |200.0 |2026-05-01 11:00:00|
|3       |Carrol  |Sochi |300.0 |2026-05-02 12:00:00|
|4       |Diana   |Moscow|400.0 |2026-05-03 09:00:00|
|5       |Egor    |Kazan |500.0 |2026-05-03 10:00:00|
+--------+--------+------+------+-------------------+

чтение из tag v1:
+--------+--------+------+------+-------------------+
|order_id|customer|city  |amount|event_ts           |
+--------+--------+------+------+-------------------+
|1       |Alice   |Moscow|100.0 |2026-05-01 10:00:00|
|2       |Bob     |Kazan |200.0 |2026-05-01 11:00:00|
|3       |Carrol  |Sochi |300.0 |2026-05-02 12:00:00|
|4       |Diana   |Moscow|400.0 |2026-05-03 09:00:00|
|5       |Egor    |Kazan |500.0 |2026-05-03 10:00:00|
+--------+--------+------+------+-------------------+



In [49]:
spark.sql(f"ALTER TABLE {CATALOG}.{NAMESPACE}.orders_branch_demo CREATE TAG v2")

spark.sql(f"SELECT name, type, snapshot_id FROM {CATALOG}.{NAMESPACE}.orders_branch_demo.refs").show(truncate=False)

spark.sql(f"ALTER TABLE {CATALOG}.{NAMESPACE}.orders_branch_demo DROP TAG v1")
spark.sql(f"SELECT name, type, snapshot_id FROM {CATALOG}.{NAMESPACE}.orders_branch_demo.refs").show(truncate=False)


+-----+------+-------------------+
|name |type  |snapshot_id        |
+-----+------+-------------------+
|main |BRANCH|5820988598573221875|
|v1   |TAG   |7322931409751006332|
|v2   |TAG   |5820988598573221875|
|audit|BRANCH|8430032323435406341|
+-----+------+-------------------+

+-----+------+-------------------+
|name |type  |snapshot_id        |
+-----+------+-------------------+
|main |BRANCH|5820988598573221875|
|v2   |TAG   |5820988598573221875|
|audit|BRANCH|8430032323435406341|
+-----+------+-------------------+



## 17. WAP (Write-Audit-Publish)

Классический паттерн: пишем в отдельную ветку `audit`, проверяем данные, и только если всё ок — публикуем в `main` через `fast_forward`. Если проверка не прошла — ветку просто удаляем, `main` не тронут.

Схема:
1. `CREATE BRANCH audit`
2. запись в audit
3. аудит на audit
4. `fast_forward('main', 'audit')`
5. `DROP BRANCH audit`


In [50]:
spark.sql(f"""
CREATE TABLE {CATALOG}.{NAMESPACE}.wap_orders (
    id INT,
    customer STRING,
    amount DOUBLE
)
USING iceberg
TBLPROPERTIES ('format-version' = '2')
""")

spark.sql(f"INSERT INTO {CATALOG}.{NAMESPACE}.wap_orders VALUES (1, 'Alice', 100.0), (2, 'Bob', 200.0)")
spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.wap_orders ORDER BY id").show(truncate=False)

spark.sql(f"ALTER TABLE {CATALOG}.{NAMESPACE}.wap_orders CREATE BRANCH audit")


+---+--------+------+
|id |customer|amount|
+---+--------+------+
|1  |Alice   |100.0 |
|2  |Bob     |200.0 |
+---+--------+------+



DataFrame[]

In [51]:
new_data = [(3, "Carrol", 300.0), (4, "Diana", 400.0)]
spark.createDataFrame(new_data, ["id", "customer", "amount"]) \
    .writeTo(f"{CATALOG}.{NAMESPACE}.wap_orders").option("branch", "audit").append()

print("main (новых строк нет):")
spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.wap_orders ORDER BY id").show(truncate=False)

print("audit (новые строки уже есть):")
spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.wap_orders VERSION AS OF 'audit' ORDER BY id").show(truncate=False)


main (новых строк нет):
+---+--------+------+
|id |customer|amount|
+---+--------+------+
|1  |Alice   |100.0 |
|2  |Bob     |200.0 |
|3  |Carrol  |300.0 |
|4  |Diana   |400.0 |
+---+--------+------+

audit (новые строки уже есть):
+---+--------+------+
|id |customer|amount|
+---+--------+------+
|1  |Alice   |100.0 |
|2  |Bob     |200.0 |
+---+--------+------+



In [52]:
audit_check = spark.sql(f"""
SELECT COUNT(*) AS bad_rows
FROM {CATALOG}.{NAMESPACE}.wap_orders VERSION AS OF 'audit'
WHERE amount IS NULL OR amount <= 0
""").first()["bad_rows"]

print("bad_rows =", audit_check)

if audit_check == 0:
    spark.sql(f"CALL {CATALOG}.system.fast_forward('{NAMESPACE}.wap_orders', 'main', 'audit')").show(truncate=False)
    print("Опубликовали audit в main")
else:
    print("Аудит не прошёл, main не публикуем")

spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.wap_orders ORDER BY id").show(truncate=False)


bad_rows = 0


IllegalArgumentException: Cannot fast-forward: main is not an ancestor of audit

In [53]:
spark.sql(f"ALTER TABLE {CATALOG}.{NAMESPACE}.wap_orders DROP BRANCH audit")
spark.sql(f"SELECT name, type, snapshot_id FROM {CATALOG}.{NAMESPACE}.wap_orders.refs").show(truncate=False)


+----+------+-------------------+
|name|type  |snapshot_id        |
+----+------+-------------------+
|main|BRANCH|5713781204617512277|
+----+------+-------------------+



## 18. CDC через MERGE

CDC-поток обычно приходит как набор записей с операцией: `I` (insert), `U` (update), `D` (delete). Применяем его к target-таблице через `MERGE INTO`. В режиме MoR на каждый UPDATE/DELETE Iceberg создаёт delete-файлы, которые потом можно убрать через `rewrite_data_files`.


In [54]:
spark.sql(f"""
CREATE TABLE {CATALOG}.{NAMESPACE}.users_cdc_demo (
    user_id INT,
    name STRING,
    email STRING,
    city STRING,
    updated_at TIMESTAMP
)
USING iceberg
TBLPROPERTIES (
    'format-version' = '2',
    'write.merge.mode' = 'merge-on-read',
    'write.update.mode' = 'merge-on-read',
    'write.delete.mode' = 'merge-on-read'
)
""")

initial = [
    (1, "Alice", "alice@example.com", "Moscow", datetime(2026, 5, 1, 10, 0)),
    (2, "Bob", "bob@example.com", "Kazan", datetime(2026, 5, 1, 10, 0)),
    (3, "Carrol", "carrol@example.com", "Sochi", datetime(2026, 5, 1, 10, 0)),
]
spark.createDataFrame(initial, ["user_id", "name", "email", "city", "updated_at"]) \
    .writeTo(f"{CATALOG}.{NAMESPACE}.users_cdc_demo").append()

spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.users_cdc_demo ORDER BY user_id").show(truncate=False)


+-------+------+------------------+------+-------------------+
|user_id|name  |email             |city  |updated_at         |
+-------+------+------------------+------+-------------------+
|1      |Alice |alice@example.com |Moscow|2026-05-01 10:00:00|
|2      |Bob   |bob@example.com   |Kazan |2026-05-01 10:00:00|
|3      |Carrol|carrol@example.com|Sochi |2026-05-01 10:00:00|
+-------+------+------------------+------+-------------------+



In [55]:
cdc_batch = [
    (1, "Alice", "alice_new@example.com", "Moscow", datetime(2026, 5, 2, 12, 0), "U"),
    (2, "Bob", "bob@example.com", "Kazan", datetime(2026, 5, 2, 12, 0), "D"),
    (4, "Diana", "diana@example.com", "Novosibirsk", datetime(2026, 5, 2, 12, 0), "I"),
    (5, "Egor", "egor@example.com", "Vladivostok", datetime(2026, 5, 2, 12, 0), "I"),
]
cdc_df = spark.createDataFrame(cdc_batch, ["user_id", "name", "email", "city", "updated_at", "op"])
cdc_df.createOrReplaceTempView("cdc_batch")
cdc_df.show(truncate=False)


+-------+-----+---------------------+-----------+-------------------+---+
|user_id|name |email                |city       |updated_at         |op |
+-------+-----+---------------------+-----------+-------------------+---+
|1      |Alice|alice_new@example.com|Moscow     |2026-05-02 12:00:00|U  |
|2      |Bob  |bob@example.com      |Kazan      |2026-05-02 12:00:00|D  |
|4      |Diana|diana@example.com    |Novosibirsk|2026-05-02 12:00:00|I  |
|5      |Egor |egor@example.com     |Vladivostok|2026-05-02 12:00:00|I  |
+-------+-----+---------------------+-----------+-------------------+---+



In [56]:
spark.sql(f"""
MERGE INTO {CATALOG}.{NAMESPACE}.users_cdc_demo t
USING (
    SELECT user_id, name, email, city, updated_at, op
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY updated_at DESC) AS rn
        FROM cdc_batch
    ) WHERE rn = 1
) s
ON t.user_id = s.user_id
WHEN MATCHED AND s.op = 'D' THEN DELETE
WHEN MATCHED AND s.op = 'U' THEN UPDATE SET
    t.name = s.name,
    t.email = s.email,
    t.city = s.city,
    t.updated_at = s.updated_at
WHEN NOT MATCHED AND s.op = 'I' THEN INSERT (user_id, name, email, city, updated_at)
    VALUES (s.user_id, s.name, s.email, s.city, s.updated_at)
""")

spark.sql(f"SELECT * FROM {CATALOG}.{NAMESPACE}.users_cdc_demo ORDER BY user_id").show(truncate=False)


+-------+------+---------------------+-----------+-------------------+
|user_id|name  |email                |city       |updated_at         |
+-------+------+---------------------+-----------+-------------------+
|1      |Alice |alice_new@example.com|Moscow     |2026-05-02 12:00:00|
|3      |Carrol|carrol@example.com   |Sochi      |2026-05-01 10:00:00|
|4      |Diana |diana@example.com    |Novosibirsk|2026-05-02 12:00:00|
|5      |Egor  |egor@example.com     |Vladivostok|2026-05-02 12:00:00|
+-------+------+---------------------+-----------+-------------------+



In [57]:
spark.sql(f"""
SELECT content, count(*) AS files, sum(record_count) AS records
FROM {CATALOG}.{NAMESPACE}.users_cdc_demo.files
GROUP BY content ORDER BY content
""").show(truncate=False)

spark.sql(f"""
SELECT committed_at, snapshot_id, operation
FROM {CATALOG}.{NAMESPACE}.users_cdc_demo.snapshots
ORDER BY committed_at
""").show(truncate=False)


+-------+-----+-------+
|content|files|records|
+-------+-----+-------+
|0      |4    |6      |
|1      |1    |2      |
+-------+-----+-------+

+-----------------------+-------------------+---------+
|committed_at           |snapshot_id        |operation|
+-----------------------+-------------------+---------+
|2026-10-04 19:52:55.825|2680993127557066491|append   |
|2026-10-04 20:00:39.006|8485680612315725171|overwrite|
+-----------------------+-------------------+---------+



Убираем delete-файлы после CDC через `rewrite_data_files`.


In [58]:
spark.sql(f"""
CALL {CATALOG}.system.rewrite_data_files(
    table => '{NAMESPACE}.users_cdc_demo'
)
""").show(truncate=False)

spark.sql(f"""
SELECT content, count(*) AS files, sum(record_count) AS records
FROM {CATALOG}.{NAMESPACE}.users_cdc_demo.files
GROUP BY content ORDER BY content
""").show(truncate=False)


+--------------------------+----------------------+---------------------+-----------------------+
|rewritten_data_files_count|added_data_files_count|rewritten_bytes_count|failed_data_files_count|
+--------------------------+----------------------+---------------------+-----------------------+
|0                         |0                     |0                    |0                      |
+--------------------------+----------------------+---------------------+-----------------------+

+-------+-----+-------+
|content|files|records|
+-------+-----+-------+
|0      |4    |6      |
|1      |1    |2      |
+-------+-----+-------+



## 19. Что проверить в конце


In [59]:
spark.sql(f"SHOW TABLES IN {CATALOG}.{NAMESPACE}").show(truncate=False)
spark.sql(f"SELECT name, type, snapshot_id FROM {CATALOG}.{NAMESPACE}.orders_branch_demo.refs").show(truncate=False)
spark.sql(f"SELECT content, count(*) AS files FROM {CATALOG}.{NAMESPACE}.mor_orders.files GROUP BY content ORDER BY content").show(truncate=False)


+---------+-----------------------+-----------+
|namespace|tableName              |isTemporary|
+---------+-----------------------+-----------+
|halltape |cow_orders             |false      |
|halltape |events_partition_demo  |false      |
|halltape |mor_orders             |false      |
|halltape |orders                 |false      |
|halltape |orders_branch_demo     |false      |
|halltape |orders_rollback_demo   |false      |
|halltape |orders_schema_evolution|false      |
|halltape |sort_orders            |false      |
|halltape |stats_demo             |false      |
|halltape |users_cdc_demo         |false      |
|halltape |wap_orders             |false      |
+---------+-----------------------+-----------+

+-----+------+-------------------+
|name |type  |snapshot_id        |
+-----+------+-------------------+
|main |BRANCH|5820988598573221875|
|v2   |TAG   |5820988598573221875|
|audit|BRANCH|8430032323435406341|
+-----+------+-------------------+

+-------+-----+
|content|files|
+

## 20. Остановка Spark


In [ ]:
spark.stop()
